In [ ]:
import pandas as pd

: 

In [ ]:
from bisect import bisect_left
from typing import List, Optional
def crossing_level(prices: List[float], sizes: List[int], target_qty: int) -> Optional[float]:
    cumulative = []
    running = 0
    for size in sizes:
        running += size
        cumulative.append(running)
    idx = bisect_left(cumulative, target_qty)
    return prices[idx] if idx < len(prices) else None

In [ ]:
from ctypes import sizeof
import heapq
from optparse import Option
from typing import Dict, Iterable, Optional, Tuple

def best_bid_offer(events: Iterable[Tuple[str, str, float, int]]) -> list[Tuple[Optional[float], Optional[float]]]:
    """Maintain the best bid and best ask for a quote stream.
    Parameters
    ----------
    events : Iterable[Tuple[str, str, float, int]]
    Sequence of (action, side, price, size) tuples.
    action is 'upsert' or 'delete'. side is 'B' or 'A'.
    Returns
    -------
    list[Tuple[Optional[float], Optional[float]]]
    Best bid and best ask after each event.
    """
    
    bids : Dict[float: int] = {}
    asks : Dict[float: int] = {}
    
    bid_heap: list[float] = []
    ask_heap: list[float] = []
    
    result = []
    
    def clean_bid() -> Optional[float]:
        while bid_heap and bids.get(-bid_heap[0], 0) <= 0:
            heapq.heappop(bid_heap)
        return -bid_heap[0] if bid_heap else None
    
    def clean_ask() -> Optional[float]:
        while ask_heap and asks.get(ask_heap[0],0) <= 0:
            heapq.heappop(ask_heap)
        return ask_heap[0] if ask_heap else None

    from action, side, price, size in events:
        book = bids if side == 'B' else asks
        heap = bid_heap if side == 'B' else ask_heap
        
        if action == 'Delete' or size <= 0:
            book['price'] = 0
        else:
            book['price'] = size
            heapq.heappush(heap, -price, if side =='B' else price)
        result.append((clean_bid(), clean_ask())
    return result
    

In [ ]:
from collections import defaultdict
from typing import Iterable, Tuple
def parent_child_match(parents: Iterable[Tuple[str, int]], children: Iterable[Tuple[str, int]]) -> list[tuple[str, bool]]:
    """Return whether child quantity fully matches each parent order."""
    child_sum = defaultdict(int)
    for parent_id, child_qty in children:
        # Sum all child quantity for each parent order.
        child_sum[parent_id] += child_qty
    result = []
    for parent_id, parent_qty in parents:
        result.append((parent_id, child_sum[parent_id] == parent_qty))
    return result
import pandas as pd


In [ ]:
import pandas as pd
def parent_child_match_pandas(parent_df: pd.DataFrame, child_df: pd.DataFrame) -> pd.DataFrame:
    """Return whether child quantity fully matches each parent order."""
    child_totals = child_df.groupby('parent_id', as_index=False)['child_qty'].sum()
    out = parent_df.merge(child_totals, on = 'parent_id', how='left')
    out['child_qty'] = out['child_qty'].filna(0)
    out['matched']=out['parent_qty'] == out['child_qty']
    return out
    

In [ ]:
from typing import Iterable, List, Tuple
def duplicate_order_ids(events: Iterable[Tuple[int, str]], horizon_secs: int) -> List[str]:
    """Return order IDs that repeat within the specified horizon.
    Parameters
    ----------
    events : Iterable[Tuple[int, str]]
    Sequence of (timestamp, order_id) tuples in time order.
    horizon_secs : int
    Maximum allowed time gap before a repeat is considered suspicious.
    Returns
    -------
    List[str]
    Duplicate IDs that violate the horizon rule.
    """
    last_seen = {}
    flagged = []
    for ts, order_id in events:
        if order_id in last_seen and ts - last_seen[order_id] <= horizon_secs:
            flagged.append(order_id)
        last_seen[order_id] = ts
    return flagged
        

In [ ]:
from collections import deque
from typing import Iterable, List, Tuple

def rolling_vwap(trades: Iterable[Tuple[int, float, int]], window_secs: int) -> List[float]:
    """Compute rolling VWAP for a time-ordered trade stream.
    Parameters
    ----------
    trades : Iterable[Tuple[int, float, int]]
    Sequence of (timestamp, price, quantity) tuples.
    window_secs : int
    Width of the rolling window in seconds.
    Returns
    -------
    List[float]
    VWAP after each trade arrives.
    """
    # Store active trades inside the current window.
    window = deque()
    
    # Tracking running totals
    running_notional = 0.0
    running_volume = 0
    result = [running_notional]
    
    for ts, price, qty in trades:
        window.append((ts,price,qty))
        running_notional += price*qty
        running_volume += qty
        while window and window[0][0] < ts-window_secs+1:
            old_ts, old_price, old_qty = window.popleft()
            running_notional -= old_price*old_qty
            running_volume -= old_qty
        result.append(running_notional/running_volume if running_volume else 0.0)
    return result

import pandas as pd
def rolling_vwap_pandas(df: pd.DataFrame, window_secs: int) -> pd.Series:
    """Compute rolling VWAP from a trade table using pandas."""
    df = df.copy()
    df['ts'] = pd.to_datetime(df['ts'])
    df = df.sort_values('ts').set_index('ts')
    
    df['notional'] = df['price']*df['qty']
    num = df['notional'].rolling(f'{window_secs}s').sum()
    den = df['volume'].rolling(f'{window_secs}s').sum()
    return num/den
        